# Production `main.py` flow test on Colab GPU

Unlike `colab_full_pipeline_test.ipynb` (which re-implements just the detect+track+OCR loop for accuracy testing), this runs the **actual production process** — `python -m app.main` — including the frame queue, HLS transcoding, and per-segment logging, on GPU. Goal: check whether GPU eliminates the CPU-contention frame drops found in `PRODUCTION_VS_OFFLINE_FRAME_REPORT.md` (production on CPU delivered only ~4/10 frames per 2s segment instead of the nominal target).

**Before running:** upload the same `app.zip`, `vechile_plate_yolov8s.pt`, and `sample_video.mp4` to your `anpr_pipeline_test` Drive folder as used by the other notebook.

**Runtime -> Change runtime type -> T4 GPU** before running.

In [ ]:
!pip install -q ultralytics opencv-python-headless paddlepaddle-gpu paddleocr==2.7.3 python-dotenv fastapi "uvicorn[standard]" requests
# numpy installed LAST, forced, no-deps -- so nothing installed above can silently
# overwrite/partially-patch its files afterward (see earlier numpy corruption issue).
!pip install -q --force-reinstall --no-deps "numpy==1.26.4"
# ffmpeg is required for HLS transcoding (app/services/hls_service.py) -- Colab images
# usually ship it already; this is a no-op if so.
!apt-get -qq install -y ffmpeg > /dev/null

**After this cell finishes:** if Colab shows a "restart runtime" prompt, click it / **Runtime -> Restart session**, then re-run this same cell once more in the fresh session. Do this restart at most once, right here — after that, go straight to the mount/stage cell below without running any more `pip install` commands.

## Mount Drive and unpack the app code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = '/content/drive/MyDrive/anpr_pipeline_test'

import os, shutil, zipfile
assert os.path.exists(f'{DRIVE_FOLDER}/app.zip'), "app.zip not found in Drive folder"

shutil.rmtree('/content/backend', ignore_errors=True)
os.makedirs('/content/backend', exist_ok=True)
with zipfile.ZipFile(f'{DRIVE_FOLDER}/app.zip') as z:
    z.extractall('/content/backend')

os.makedirs('/content/backend/models', exist_ok=True)

MODEL_NAME = 'vechile_plate_yolov8s.pt'
VIDEO_NAME = 'sample_video.mp4'

shutil.copy(f'{DRIVE_FOLDER}/{MODEL_NAME}', f'/content/backend/models/{MODEL_NAME}')
shutil.copy(f'{DRIVE_FOLDER}/{VIDEO_NAME}', f'/content/backend/{VIDEO_NAME}')

print("App code + model + video staged.")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
# app/config.py resolves DEVICE from torch.cuda.is_available() at import time,
# so PlateDetector picks up the GPU automatically once this is True.

## Force PaddleOCR onto CPU only (avoids the cuDNN clash documented in ANPR_COLAB_VS_CPU_REPORT.md)

This patches `app/ocr/plate_reader.py`'s `PlateReader.__init__` on disk, since `main.py` constructs
everything internally (no notebook-side object to patch after the fact like the other notebook does).

In [ ]:
path = "/content/backend/app/ocr/plate_reader.py"
src = open(path).read()
old = 'self._ocr = PaddleOCR(use_angle_cls=False, lang=lang, show_log=False)'
new = 'self._ocr = PaddleOCR(use_angle_cls=False, lang=lang, show_log=False, use_gpu=False)'
assert old in src, "PlateReader.__init__ line not found -- check app.zip is up to date"
open(path, "w").write(src.replace(old, new))
print("Patched PlateReader to force OCR onto CPU (plate detection stays on GPU).")

## Run `python -m app.main` in the background, trigger start, wait for completion

No `DATABASE_URL` is set, so results are only printed/logged, not persisted to Postgres --
fine for this frame-delivery test.

In [ ]:
import subprocess, time, os, requests

os.chdir("/content/backend")
log_path = "/content/backend/run.log"
if os.path.exists(log_path):
    os.remove(log_path)

log_file = open(log_path, "w")
proc = subprocess.Popen(
    ["python", "-m", "app.main", "--source", f"/content/backend/{VIDEO_NAME}", "--camera-id", "cam01"],
    stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"Started main.py, pid={proc.pid}")

# Wait for the API server + pipeline warmup to be ready.
for _ in range(60):
    time.sleep(1)
    if os.path.exists(log_path) and "Ready" in open(log_path).read():
        break
print("Ready line seen (or timed out) -- tailing log so far:")
print(open(log_path).read()[-1500:])

In [ ]:
resp = requests.post("http://localhost:8765/api/cameras/cam01/start")
print(resp.status_code, resp.text)

In [ ]:
# Poll until the cycle finishes (video file exhausted) or a generous timeout elapses.
import time

MAX_WAIT_S = 600
t0 = time.time()
while time.time() - t0 < MAX_WAIT_S:
    time.sleep(3)
    content = open(log_path).read()
    if "Cycle finished" in content:
        print(f"Cycle finished after {time.time() - t0:.1f}s wait.")
        break
else:
    print("Timed out waiting for 'Cycle finished' -- check the log manually below.")

print(content[-2000:])

## Check frames-per-segment (the actual question: does GPU deliver the nominal 10 frames/2s-segment?)

In [ ]:
import re

content = open(log_path).read()
counts = [int(m) for m in re.findall(r"Segment \d+ done: (\d+) frames", content)]
# Each log line is duplicated (plain + bracketed formatter) in this project's logger setup --
# dedupe by taking every other match if that's the case here too.
if len(counts) % 2 == 0 and counts[::2] == counts[1::2]:
    counts = counts[::2]

print(f"Segments: {len(counts)}")
print(f"Per-segment frame counts: {counts}")
print(f"Total frames delivered: {sum(counts)}")
print(f"Average frames/segment: {sum(counts)/len(counts):.2f}")
print(f"Nominal target (2s x 5fps): 10")

dropped = re.findall(r"dropped_total=(\d+)", content)
if dropped:
    print(f"Final dropped_frames (frame-queue drops): {dropped[-1] if len(dropped)%2==0 is False else dropped[-2]}")

## Fetch validated plates + vehicle_count via the API, for accuracy comparison

In [ ]:
import json

camera_info = requests.get("http://localhost:8765/api/cameras/cam01").json()
print(json.dumps(camera_info, indent=2))

plates_resp = requests.get("http://localhost:8765/api/detections/cam01/plates").json()
with open("/content/colab_main_plates.json", "w") as f:
    json.dump(plates_resp, f, indent=2)

accepted = [r for r in plates_resp["results"] if r["status"] == "accepted"]
distinct_plates = sorted(set(r["plate"] for r in accepted))
print(f"\nDistinct accepted plates ({len(distinct_plates)}):")
for p in distinct_plates:
    print(" ", p)

from google.colab import files
files.download("/content/colab_main_plates.json")

## Compare against the CPU production run

Back on your own computer, with `backend/run.log` (the CPU production run from
`PRODUCTION_VS_OFFLINE_FRAME_REPORT.md`) and this notebook's printed frames-per-segment /
distinct-plates numbers side by side, check:
1. Does GPU's avg frames/segment get closer to the nominal 10 (vs CPU's ~4)?
2. Does the distinct-plates count/overlap match the CPU run, confirming GPU doesn't change *what* gets read, only throughput?